# Users — Bronze to Silver

This notebook transforms raw user account records from the **bronze** layer into a curated **silver** Delta table for downstream reporting and analytics.

It works at the **user account level**, meaning each row represents a unique user profile from the auction platform. The notebook standardizes names, converts account creation timestamps, and loads the cleaned records into `company_db.silver.users`.

## What this notebook is doing

This sheet provides a breakdown of platform users, including:
* the user ID tied to the account
* the user's full name and email address
* company, billing, and shipping details
* account status and bid limit
* account creation timestamp
* source lineage fields used for traceability

## Main transformations

1. Reads raw user-level records from `company_db.bronze.users`.
2. Builds a cleaned `full_name` from first and last name.
3. Renames `user_#` to `user_id` for analytics-friendly naming.
4. Parses `date_created` into a proper timestamp.
5. Converts the timestamp to the reporting timezone.
6. Creates the target Delta table in the silver layer.
7. Registers the transformed DataFrame as a temporary view.
8. Merges transformed records into silver so existing users are updated and new users are inserted.

## Source and target

* **Source:** `company_db.bronze.users`
* **Target:** `company_db.silver.users`
* **Storage location:** `abfss://auction@company.dfs.core.windows.net/silver/users`

## Data dictionary

| Column | Meaning |
| --- | --- |
| `user_id` | Unique user identifier from the source platform. |
| `first_name` | User first name. |
| `last_name` | User last name. |
| `full_name` | Proper-cased full name built from first and last name. |
| `items_won` | Source field tracking items won by the user. |
| `highest_won` | Source field showing the highest-value win associated with the user. |
| `email_address` | Email address tied to the user account. |
| `company_name` | Company name associated with the user profile. |
| `company_url` | Company website URL associated with the user profile. |
| `billing_firstname` | Billing contact first name. |
| `billing_lastname` | Billing contact last name. |
| `billing_companyname` | Billing company name. |
| `billing_address_1` | Primary billing street address. |
| `billing_address_2` | Secondary billing address line. |
| `billing_city` | Billing city. |
| `billing_state` | Billing state or province. |
| `billing_country` | Billing country. |
| `billing_zip` | Billing ZIP or postal code. |
| `billing_phone` | Billing phone number. |
| `shipping_firstname` | Shipping contact first name. |
| `shipping_lastname` | Shipping contact last name. |
| `shipping_companyname` | Shipping company name. |
| `shipping_address_1` | Primary shipping street address. |
| `shipping_address_2` | Secondary shipping address line. |
| `shipping_city` | Shipping city. |
| `shipping_state` | Shipping state or province. |
| `shipping_country` | Shipping country. |
| `shipping_zip` | Shipping ZIP or postal code. |
| `shipping_phone` | Shipping phone number. |
| `status` | Operational status of the user account. |
| `bid_limit` | Bid limit assigned to the user. |
| `customfields` | Additional source attributes stored as raw custom fields. |
| `date_created` | Timestamp when the user account was created, converted to the reporting timezone. |
| `path` | Source file or ingestion path used for traceability. |
| `ingestion_timestamp` | Timestamp when the raw record was ingested. |
| `created_at` | Timestamp when the record was first inserted into silver. |
| `updated_at` | Timestamp when the record was last updated in silver. |

## Business interpretation

* A row represents one unique user account from the auction platform.
* `status` shows the operational account state.
* `bid_limit` supports user segmentation and bidding controls analysis.
* This silver table supports customer analysis, user profile enrichment, operational reporting, and downstream joins to registrations, bids, invoices, and auction activity.

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Import PySpark SQL functions used for user profile transformations and timestamp handling.
from pyspark.sql import functions as F
from pyspark.sql.functions import split

In [ ]:
# Read the raw registered users table from the bronze layer.
# This DataFrame is the source for all registration-level cleaning and business transformations.
users = spark.table(f'{catalog}.bronze.users').filter(F.col('batch_id')==batch_id)

In [ ]:
from src.transformations.users import transform_users

In [ ]:
users = transform_users(users)

In [ ]:
# Register the transformed DataFrame as a temporary view so it can be used by the SQL MERGE statement.
users.createOrReplaceTempView('users_view')

In [ ]:
spark.sql(
f"""CREATE TABLE IF NOT EXISTS {catalog}.silver.users (
    user_id INT,
    first_name STRING,
    last_name STRING,
    full_name STRING,
    items_won STRING,
    highest_won STRING,
    email_address STRING,
    company_name STRING,
    company_url STRING,
    billing_firstname STRING,
    billing_lastname STRING,
    billing_companyname STRING,
    billing_address_1 STRING,
    billing_address_2 STRING,
    billing_city STRING,
    billing_state STRING,
    billing_country STRING,
    billing_zip STRING,
    billing_phone STRING,
    shipping_firstname STRING,
    shipping_lastname STRING,
    shipping_companyname STRING,
    shipping_address_1 STRING,
    shipping_address_2 STRING,
    shipping_city STRING,
    shipping_state STRING,
    shipping_country STRING,
    shipping_zip STRING,
    shipping_phone STRING,
    status STRING,
    bid_limit INT,
    customfields STRING,
    date_created TIMESTAMP,
    path STRING,
    ingestion_timestamp TIMESTAMP,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    batch_id STRING
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/silver/users';""")

In [ ]:
# Upsert transformed user account records into the silver users table.
# Existing rows are updated by user_id, and new rows are inserted.

spark.sql(
f"""MERGE INTO {catalog}.silver.users AS target
USING users_view AS source
ON target.user_id = source.user_id

WHEN MATCHED
AND target.batch_id >= source.batch_id  
THEN
UPDATE SET
    target.first_name = source.first_name,
    target.last_name = source.last_name,
    target.full_name = source.full_name,
    target.items_won = source.items_won,
    target.highest_won = source.highest_won,
    target.email_address = source.email_address,
    target.company_name = source.company_name,
    target.company_url = source.company_url,
    target.billing_firstname = source.billing_firstname,
    target.billing_lastname = source.billing_lastname,
    target.billing_companyname = source.billing_companyname,
    target.billing_address_1 = source.billing_address_1,
    target.billing_address_2 = source.billing_address_2,
    target.billing_city = source.billing_city,
    target.billing_state = source.billing_state,
    target.billing_country = source.billing_country,
    target.billing_zip = source.billing_zip,
    target.billing_phone = source.billing_phone,
    target.shipping_firstname = source.shipping_firstname,
    target.shipping_lastname = source.shipping_lastname,
    target.shipping_companyname = source.shipping_companyname,
    target.shipping_address_1 = source.shipping_address_1,
    target.shipping_address_2 = source.shipping_address_2,
    target.shipping_city = source.shipping_city,
    target.shipping_state = source.shipping_state,
    target.shipping_country = source.shipping_country,
    target.shipping_zip = source.shipping_zip,
    target.shipping_phone = source.shipping_phone,
    target.status = source.status,
    target.bid_limit = source.bid_limit,
    target.customfields = source.customfields,
    target.date_created = source.date_created,
    target.path = source.path,
    target.ingestion_timestamp = source.ingestion_timestamp,
    target.updated_at = current_timestamp(),
    target.batch_id  = source.batch_id
WHEN NOT MATCHED THEN
INSERT (
    user_id,
    first_name,
    last_name,
    full_name,
    items_won,
    highest_won,
    email_address,
    company_name,
    company_url,
    billing_firstname,
    billing_lastname,
    billing_companyname,
    billing_address_1,
    billing_address_2,
    billing_city,
    billing_state,
    billing_country,
    billing_zip,
    billing_phone,
    shipping_firstname,
    shipping_lastname,
    shipping_companyname,
    shipping_address_1,
    shipping_address_2,
    shipping_city,
    shipping_state,
    shipping_country,
    shipping_zip,
    shipping_phone,
    status,
    bid_limit,
    customfields,
    date_created,
    path,
    ingestion_timestamp,
    created_at,
    updated_at,
    batch_id
)
VALUES (
    source.user_id,
    source.first_name,
    source.last_name,
    source.full_name,
    source.items_won,
    source.highest_won,
    source.email_address,
    source.company_name,
    source.company_url,
    source.billing_firstname,
    source.billing_lastname,
    source.billing_companyname,
    source.billing_address_1,
    source.billing_address_2,
    source.billing_city,
    source.billing_state,
    source.billing_country,
    source.billing_zip,
    source.billing_phone,
    source.shipping_firstname,
    source.shipping_lastname,
    source.shipping_companyname,
    source.shipping_address_1,
    source.shipping_address_2,
    source.shipping_city,
    source.shipping_state,
    source.shipping_country,
    source.shipping_zip,
    source.shipping_phone,
    source.status,
    source.bid_limit,
    source.customfields,
    source.date_created,
    source.path,
    source.ingestion_timestamp,
    current_timestamp(),
    current_timestamp(),
    source.batch_id
);"""
)

## Analyst notes

Use this table when you need user-level platform reporting such as:
* total active user profiles
* user account status analysis
* bid limit segmentation
* user geography based on billing or shipping details
* customer profile enrichment for downstream auction analysis
* joins to registrations, bids, invoices, and payment activity

Because the notebook stores one row per user account, it is well suited for customer reporting, operational review, and silver-to-gold aggregations.

## Business usage notes

This silver table is the curated user dimension for platform customers.

Use it to answer questions such as:
* how many user accounts exist in the platform
* which users are active or inactive
* how user geography is distributed across billing and shipping locations
* which customer profiles can be joined to registrations, bids, wins, and invoices

This table is best treated as a user-level master profile table for downstream analytics.